# 00 — Overview: Enterprise Cloud Automation & Infrastructure Platform

Projeto de portfólio (Yuri Fernando Dubbern), construído para demonstrar as
competências pedidas por uma vaga pública de **Enterprise Automation
Engineer**: Terraform (IaC) como núcleo, Ansible + PowerShell para
configuração Linux/Windows, GitHub Actions para CI/CD, Python para
orquestração e self-healing, MySQL para registro de execuções/incidentes, e
um dashboard de observabilidade.

> Não é software da Caterpillar nem a representa. Origem completa do escopo:
> [`escopo.md`](../escopo.md). Registro central de decisões e sessões:
> [`PROJECT_LOG.md`](../PROJECT_LOG.md).

Este notebook é o ponto de entrada: localiza a raiz do projeto, mostra a
arquitetura e resume o que cada notebook seguinte demonstra — todos
executáveis de ponta a ponta, sem custo real de AWS e sem precisar de
credenciais.

In [1]:
import sys
from pathlib import Path

def find_project_root(start: Path) -> Path:
    """Walk upward from `start` until a directory containing PROJECT_LOG.md
    is found. Works whether the notebook is executed from notebooks/ (the
    normal case) or from the repo root."""
    p = start.resolve()
    for _ in range(8):
        if (p / "PROJECT_LOG.md").exists():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise FileNotFoundError("Could not locate project root (PROJECT_LOG.md not found upward from %s)" % start)

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: G:\Outros computadores\Meu computador\Controle Base\Projetos, Robos e Automação\Projetos Git\Projetos Extras (Portfolio)\Caterpillar (Terminar)


## Objetivo do projeto (extraído de PROJECT_LOG.md)

In [2]:
log_text = (PROJECT_ROOT / "PROJECT_LOG.md").read_text(encoding="utf-8")
start = log_text.find("## Objetivo do projeto")
end = log_text.find("## Decisões-chave")
print(log_text[start:end].strip() if start != -1 else "Seção não encontrada.")

## Objetivo do projeto

Plataforma de portfólio (não é software da Caterpillar, nem a representa)
construída para demonstrar competências de **Enterprise Automation Engineer**:
Terraform (IaC) como núcleo, Ansible + PowerShell para configuração
Linux/Windows, GitHub Actions para CI/CD, Python para orquestração e
troubleshooting/self-healing, MySQL para registro de execuções/incidentes, e
um dashboard simples de observabilidade. Origem completa do escopo em
[`escopo.md`](escopo.md).

Link do repositório: preencher após criação via `gh repo create` (ver seção
Pendências).


## Arquitetura

```
                        GitHub
                          |
                    Pull Request
                          |
                          v
                  GitHub Actions
               +----------+----------+
               |                     |
        terraform fmt/validate   Security Scan
               |               (checkov / tflint)
               v                     |
          terraform plan             |
               |                     |
               +----------+----------+
                          v
                    Terraform Apply *
                          |
                          v
                         AWS
          +---------------+----------------+
          |               |                |
         VPC             EC2              RDS
          |               |                |
     Subnets / SG      Linux/Windows     MySQL
                          |
                +---------+---------+
                v                   v
             Ansible            PowerShell
                |                   |
        Linux configuration    Windows automation
                |                   |
                +---------+---------+
                          v
              Automation Controller (Python)
                          |
                +---------+---------+
                v         v         v
          Health Check  Inventory  Remediation
           (RCA/self-healing, boto3, MySQL)
                          |
                          v
                Bootstrap Dashboard (JS/Bootstrap)
```

`*` `terraform apply` é sempre uma decisão manual e explícita do dono do
projeto (nunca roda automaticamente em CI) — ver Notebook 01.

## Roadmap de versões (ver [`CHANGELOG.md`](../CHANGELOG.md))

| Versão | Nome | Foco |
|--------|------|------|
| v0.1 | Terraform Foundation | Módulos IaC: networking, compute, database, iam, security, monitoring |
| v0.2 | Automation | Ansible (Linux) + PowerShell (Windows) + Python controller |
| v0.3 | DevOps / CI-CD | GitHub Actions, checkov, tflint, pre-commit, testes |
| v0.4 | Enterprise | Inventário AWS (boto3/moto), MySQL, dashboard de observabilidade |
| v1.0 | Self-Healing | Health check -> diagnóstico -> remediação -> validação -> relatório de incidente |

## O que cada notebook seguinte demonstra

| Notebook | Demonstra |
|----------|-----------|
| [`01_terraform_foundation.ipynb`](01_terraform_foundation.ipynb) | `terraform fmt -check` / `terraform validate` nos 6 módulos (networking, compute, database, iam, security, monitoring), sem `apply` |
| [`02_automation_linux_windows.ipynb`](02_automation_linux_windows.ipynb) | Módulos `automation/troubleshooting/*.py` chamados diretamente em Python (health/disk/service check), estrutura das roles Ansible e do módulo PowerShell InfraOps |
| [`03_cicd_security.ipynb`](03_cicd_security.ipynb) | Parse dos workflows `.github/workflows/*.yml` (jobs/steps) e explicação de checkov/tflint/pre-commit |
| [`04_enterprise_observability.ipynb`](04_enterprise_observability.ipynb) | `automation/inventory/aws_inventory.py` rodando contra AWS mockado com `moto` (zero custo), schema MySQL e como o dashboard consome os dados |
| [`05_self_healing_incident.ipynb`](05_self_healing_incident.ipynb) | Fluxo completo de self-healing simulado reproduzindo o "Incident #017" do escopo |

## Estrutura de diretórios (estado atual)

In [3]:
import os

dirs_of_interest = [
    "terraform/modules",
    "terraform/environments",
    "ansible/roles",
    "powershell",
    "automation",
    "dashboard",
    "database",
    ".github/workflows",
    "docs/logs",
]

for rel in dirs_of_interest:
    p = PROJECT_ROOT / rel
    if not p.exists():
        print(f"{rel:35s} -> (ainda não existe)")
        continue
    entries = sorted(str(x.relative_to(p)) for x in p.rglob("*") if x.is_file())
    print(f"{rel:35s} -> {len(entries)} arquivo(s)")
    for e in entries[:8]:
        print(f"    {e}")
    if len(entries) > 8:
        print(f"    ... (+{len(entries) - 8})")

terraform/modules                   -> 18 arquivo(s)
    compute\main.tf
    compute\outputs.tf
    compute\variables.tf
    database\main.tf
    database\outputs.tf
    database\variables.tf
    iam\main.tf
    iam\outputs.tf
    ... (+10)


terraform/environments              -> 30 arquivo(s)
    dev\.terraform.lock.hcl
    dev\.terraform\modules\modules.json
    dev\.terraform\providers\registry.terraform.io\hashicorp\aws\5.100.0\windows_amd64\LICENSE.txt
    dev\.terraform\providers\registry.terraform.io\hashicorp\aws\5.100.0\windows_amd64\terraform-provider-aws_v5.100.0_x5.exe
    dev\backend.tf
    dev\main.tf
    dev\outputs.tf
    dev\providers.tf
    ... (+22)


ansible/roles                       -> 13 arquivo(s)
    base\defaults\main.yml
    base\handlers\main.yml
    base\meta\main.yml
    base\tasks\firewall.yml
    base\tasks\hardening.yml
    base\tasks\main.yml
    base\tasks\packages.yml
    base\tasks\users.yml
    ... (+5)


powershell                          -> 0 arquivo(s)


automation                          -> 33 arquivo(s)
    __init__.py
    __pycache__\__init__.cpython-310.pyc
    cli\__init__.py
    cli\__main__.py
    cli\main.py
    inventory\__init__.py
    inventory\__pycache__\__init__.cpython-310.pyc
    inventory\__pycache__\aws_inventory.cpython-310.pyc
    ... (+25)
dashboard                           -> 5 arquivo(s)
    README.md
    css\style.css
    index.html
    js\app.js
    js\data.example.json


database                            -> 5 arquivo(s)
    .env.example
    README.md
    docker-compose.yml
    schema.sql
    seed_data.sql
.github/workflows                   -> 5 arquivo(s)
    ansible-lint.yml
    powershell-tests.yml
    python-tests.yml
    security-scan.yml
    terraform-ci.yml


docs/logs                           -> 3 arquivo(s)
    automation-python.md
    dashboard-tests.md
    terraform.md


## Como rodar

Todos os notebooks usam apenas dados/mocks locais (nenhuma chamada real à
AWS, sem necessidade de Docker/MySQL rodando) e têm fallback gracioso com
mensagens claras quando alguma dependência opcional não está disponível.

```bash
C:\Users\Yuri_\AppData\Local\Programs\Python\Python310\python.exe -m jupyter nbconvert --to notebook --execute notebooks/00_overview.ipynb
```

Ou abra no Jupyter Lab/Notebook normalmente. Detalhes completos em
[`docs/logs/notebooks-docs.md`](../docs/logs/notebooks-docs.md).